# Evaluation & Statistics

MAE/RMSE/R2/MAPE, stratified error breakdowns (e.g. by job-size bucket, not just a single aggregate number), multi-seed paired significance testing (statistical testing of whether one model's advantage over another, established in notebook `05d`'s seed sweeps, is significant rather than seed noise), and a rolling-window temporal-drift check for F-DATA (whether the rolling-window user-history feature degrades or drifts over the dataset's time span).

The paired significance testing is implemented below: a user-clustered paired bootstrap of the R2 difference, run for every pairwise model comparison (RandomForest vs XGBoost, RandomForest vs LightGBM, XGBoost vs LightGBM) within each of the six target/dataset combinations. The stratified error breakdowns and the temporal-drift check are not yet built.

See the conceptualization plan and EXPERIMENT_TRACKER.md for full context.

In [1]:
import sys
sys.path.append("..")

from src import config, features, metrics, plotting, models, roofline

## Paired significance testing: a user-clustered bootstrap of the R2 difference

Notebook `05d` asks one question: does refitting an already-tuned model with a different
training seed change the outcome? It repeats each final fit at 3-5 seeds and checks whether the
resulting R2 ranges for two models overlap. That question is about training-seed variance, and
its answer comes from a handful of numbers (3-5 per model) -- far too few for a seed-level
significance test to ever be useful. A two-sided Wilcoxon signed-rank test on seed-level R2
values can reach p < 0.05 only with at least 6 paired seeds; at n=5 the smallest achievable
two-sided p-value is 2 x 0.5^5 = 0.0625, and at n=3 it is 2 x 0.5^3 = 0.25 -- both already above
0.05 before a single number is looked at. Underpowered by construction, not by the data, so it
was ruled out rather than run.

This notebook asks a different question: given the two ALREADY-FITTED models from a single
seed, is the performance gap observed on the test set distinguishable from what finite-test-set
sampling alone could produce? The test sets here range from 54,093 rows (PM100) to 6,992,514 rows
(F-DATA) -- real statistical power exists at that scale, just not at the 3-5-seed scale. The two
questions are complementary, not interchangeable, and are reported side by side below rather than
combined into a single number: a gap can be real-on-this-test-set (this notebook) yet not
survive a training-seed change (05d), or vice versa.

**Method.** Resample test-set USERS, not rows, with replacement -- preserving whatever
row-to-row correlation exists within a user's own jobs, which plain row-level bootstrapping would
destroy. Per-user sufficient statistics (row count, sum of y, sum of y^2, sum of squared residual
for each of the two models being compared) are precomputed once per model pair; every bootstrap
iteration then reconstructs R2 for both models algebraically from these sums alone, without
touching individual rows -- the cost of one iteration is O(n_users), not O(n_rows), which is what
makes 2000 iterations on a 7-million-row F-DATA test set take milliseconds rather than minutes.
y is centered (subtract the uncentered global mean) before any sum is accumulated, purely for
floating-point stability at F-DATA's scale, where sum(y^2) and (sum(y))^2/n can both be huge and
nearly equal -- centering does not change R2, which is invariant to a constant shift in y.

Both raw-space and log-space delta-R2 are reported for every comparison: PM100 duration's own
results have disagreed between the two spaces before, and there is no reason to assume the other
five combinations wouldn't too.


In [2]:
import sys
sys.path.append("..")

import time

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

from src import features

INTERIM_DIR = "../data/interim"
RESULTS_DIR = "../results"

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 60)

### Core machinery

The same four functions used for the full sweep (`scripts/notebook08_paired_bootstrap/bootstrap_lib.py`,
which also holds the sweep driver itself), reproduced here so this notebook is readable
standalone.

In [3]:
def compute_per_user_stats(user_ids, y_true, pred_a, pred_b):
    """y_true, pred_a, pred_b must be in the SAME space (all-raw or all-log) and the same row
    order as user_ids. Returns per-user sufficient statistics plus the centering constant used."""
    y_true = np.asarray(y_true, dtype=float)
    pred_a = np.asarray(pred_a, dtype=float)
    pred_b = np.asarray(pred_b, dtype=float)
    center = float(np.mean(y_true))
    y_c = y_true - center

    users, inverse = np.unique(np.asarray(user_ids), return_inverse=True)
    n_users = len(users)

    n_u = np.bincount(inverse, minlength=n_users).astype(float)
    sy_u = np.bincount(inverse, weights=y_c, minlength=n_users)
    sy2_u = np.bincount(inverse, weights=y_c ** 2, minlength=n_users)
    ssa_u = np.bincount(inverse, weights=(y_c - (pred_a - center)) ** 2, minlength=n_users)
    ssb_u = np.bincount(inverse, weights=(y_c - (pred_b - center)) ** 2, minlength=n_users)

    return dict(users=users, n_u=n_u, sy_u=sy_u, sy2_u=sy2_u, ssa_u=ssa_u, ssb_u=ssb_u,
                center=center, n_users=n_users, n_rows=len(y_true))


def r2_from_sufficient_stats(n_total, sy_total, sy2_total, ss_res_total):
    ss_tot = sy2_total - (sy_total ** 2) / n_total
    return 1.0 - ss_res_total / ss_tot


def run_cluster_bootstrap(stats, n_iterations=2000, seed=0):
    """Vectorized user-cluster bootstrap: O(n_users) per iteration, not O(n_rows)."""
    rng = np.random.default_rng(seed)
    n_users = stats["n_users"]
    idx = rng.integers(0, n_users, size=(n_iterations, n_users))

    r2_a = np.empty(n_iterations)
    r2_b = np.empty(n_iterations)
    for it in range(n_iterations):
        counts = np.bincount(idx[it], minlength=n_users).astype(float)
        n_total = counts @ stats["n_u"]
        sy_total = counts @ stats["sy_u"]
        sy2_total = counts @ stats["sy2_u"]
        ssa_total = counts @ stats["ssa_u"]
        ssb_total = counts @ stats["ssb_u"]
        r2_a[it] = r2_from_sufficient_stats(n_total, sy_total, sy2_total, ssa_total)
        r2_b[it] = r2_from_sufficient_stats(n_total, sy_total, sy2_total, ssb_total)
    return r2_a, r2_b


def run_rowlevel_bootstrap(user_ids, y_true, pred_a, pred_b, n_iterations=2000, seed=0):
    """Brute-force reference implementation, used only as a one-time cross-check below."""
    user_ids = np.asarray(user_ids)
    y_true = np.asarray(y_true, dtype=float)
    pred_a = np.asarray(pred_a, dtype=float)
    pred_b = np.asarray(pred_b, dtype=float)
    users = np.unique(user_ids)
    n_users = len(users)
    rows_by_user = {u: np.where(user_ids == u)[0] for u in users}

    rng = np.random.default_rng(seed)
    r2_a = np.empty(n_iterations)
    r2_b = np.empty(n_iterations)
    for it in range(n_iterations):
        drawn = rng.choice(users, size=n_users, replace=True)
        row_idx = np.concatenate([rows_by_user[u] for u in drawn])
        r2_a[it] = r2_score(y_true[row_idx], pred_a[row_idx])
        r2_b[it] = r2_score(y_true[row_idx], pred_b[row_idx])
    return r2_a, r2_b


def top5_user_share(stats):
    """Share of total rows and total (centered) sum-of-squares held by the 5 heaviest users."""
    order = np.argsort(-stats["n_u"])[:5]
    row_share = stats["n_u"][order].sum() / stats["n_rows"]
    ss_share = stats["sy2_u"][order].sum() / stats["sy2_u"].sum()
    return float(row_share), float(ss_share)

### Validation, on PM100 power, before trusting this on the other five combinations

Two checks, run before any of the 18 real comparisons: (1) does reconstructing R2 from the
per-user sufficient statistics, applied to the full unresampled test set, reproduce the actual
reported R2 exactly; (2) does the fast sufficient-statistics bootstrap agree with a brute-force
row-level cluster bootstrap that resamples and scores actual rows every iteration. PM100 power is
used because all three of its models are cheap to refit (no multi-hour fits to wait on) and its
headline R2 values are independently known from `results/results_ledger.csv`.

In [4]:
PM100_PATH = "../data/raw/pm100/pm100_job_table.parquet"

from src import splits

t0 = time.perf_counter()
pm100 = pd.read_parquet(PM100_PATH)
pm100 = features.filter_completed_jobs(pm100, "pm100")
_, pm100_test_df = splits.chronological_split(pm100, "pm100")

prep = joblib.load(f"{INTERIM_DIR}/pm100_power_prepared.joblib")
X_train, X_test = prep["X_train"], prep["X_test"]
y_train_log, y_test_raw = prep["y_train"], prep["y_test_raw"]
best_params = prep["best_params"]

assert list(pm100_test_df.index) == list(X_test.index), "index alignment failed"
user_ids = pm100_test_df.loc[X_test.index, "user_id"].to_numpy()
print(f"index alignment confirmed: {len(np.unique(user_ids))} unique test users, "
      f"{len(user_ids):,} rows ({time.perf_counter() - t0:.1f}s)")

pred_log = {}
for model_name in ["RandomForest", "XGBoost", "LightGBM"]:
    params = best_params[model_name]
    if model_name == "RandomForest":
        m = RandomForestRegressor(**params, n_jobs=-1, random_state=0)
    elif model_name == "XGBoost":
        m = XGBRegressor(**params, tree_method="hist", n_jobs=-1, random_state=0)
    else:
        m = LGBMRegressor(**params, bagging_freq=1, n_jobs=-1, random_state=0, verbose=-1)
    m.fit(X_train, y_train_log)
    pred_log[model_name] = np.asarray(m.predict(X_test), dtype=float)
print("seed-0 refit done for all three models")

[pm100] excluded 22.0% of jobs as non-completed (50928 / 231238)
index alignment confirmed: 250 unique test users, 54,093 rows (2.0s)


seed-0 refit done for all three models


#### Check 1: sufficient-statistics R2 reconstruction vs. the real reported R2

In [5]:
real_r2_raw = {"RandomForest": 0.901822, "XGBoost": 0.924327, "LightGBM": 0.923747}  # seed 0,
# results_ledger.csv, raw-space, is_seed_sweep, variant="seed_sweep" (matches "adopted" to 4dp)

rows = []
for model_name in ["RandomForest", "XGBoost", "LightGBM"]:
    pred_raw = features.inverse_transform_target(pred_log[model_name])
    stats = compute_per_user_stats(user_ids, y_test_raw, pred_raw, pred_raw)
    recon = r2_from_sufficient_stats(stats["n_u"].sum(), stats["sy_u"].sum(),
                                      stats["sy2_u"].sum(), stats["ssa_u"].sum())
    diff = abs(recon - real_r2_raw[model_name])
    rows.append(dict(model=model_name, reconstructed=recon, real=real_r2_raw[model_name],
                      diff=diff, pass_=diff < 1e-6))
check1_df = pd.DataFrame(rows)
display(check1_df)
assert check1_df["pass_"].all(), "sufficient-statistics reconstruction did not match the real R2"
print("Check 1: PASS for all three models (diffs are float-precision noise from n_jobs=-1 "
      "multi-threaded fit/predict, not a reconstruction error -- confirmed not bit-reproducible "
      "even refitting twice at the same random_state).")

,model,reconstructed,real,diff,pass_
0,RandomForest,0.901822,0.901822,4.268574e-07,True
1,XGBoost,0.924327,0.924327,1.773180e-07,True
2,LightGBM,0.923747,0.923747,4.761737e-07,True


Check 1: PASS for all three models (diffs are float-precision noise from n_jobs=-1 multi-threaded fit/predict, not a reconstruction error -- confirmed not bit-reproducible even refitting twice at the same random_state).


#### Check 2: fast (sufficient-statistics) vs. brute-force row-level cluster bootstrap\n\nXGBoost vs LightGBM, raw space, 2000 iterations both ways.

In [6]:
pred_a_raw = features.inverse_transform_target(pred_log["XGBoost"])
pred_b_raw = features.inverse_transform_target(pred_log["LightGBM"])

stats_ab = compute_per_user_stats(user_ids, y_test_raw, pred_a_raw, pred_b_raw)
t0 = time.perf_counter()
fast_r2a, fast_r2b = run_cluster_bootstrap(stats_ab, n_iterations=2000, seed=42)
t_fast = time.perf_counter() - t0
fast_delta = fast_r2a - fast_r2b

t0 = time.perf_counter()
brute_r2a, brute_r2b = run_rowlevel_bootstrap(user_ids, y_test_raw, pred_a_raw, pred_b_raw,
                                               n_iterations=2000, seed=42)
t_brute = time.perf_counter() - t0
brute_delta = brute_r2a - brute_r2b

check2_df = pd.DataFrame({
    "method": ["fast (sufficient-stats)", "brute-force (row-level)"],
    "mean_delta": [fast_delta.mean(), brute_delta.mean()],
    "std_delta": [fast_delta.std(), brute_delta.std()],
    "ci_99_lo": [np.percentile(fast_delta, 0.5), np.percentile(brute_delta, 0.5)],
    "ci_99_hi": [np.percentile(fast_delta, 99.5), np.percentile(brute_delta, 99.5)],
    "elapsed_s": [t_fast, t_brute],
})
display(check2_df)
mean_diff = abs(fast_delta.mean() - brute_delta.mean())
std_diff = abs(fast_delta.std() - brute_delta.std())
print(f"mean(delta) difference: {mean_diff:.6f}   std(delta) difference: {std_diff:.6f}   "
      f"speedup: {t_brute / t_fast:.1f}x")
assert mean_diff < 0.005 and std_diff < 0.005, "fast and brute-force bootstraps disagree"
print("Check 2: PASS -- fast and brute-force bootstraps agree to 6 decimal places.")

,method,mean_delta,std_delta,ci_99_lo,ci_99_hi,elapsed_s
0,fast (sufficient-stats),-0.000287,0.00255,-0.009987,0.003242,0.020506
1,brute-force (row-level),-0.000287,0.00255,-0.009987,0.003242,3.897644


mean(delta) difference: 0.000000   std(delta) difference: 0.000000   speedup: 190.1x
Check 2: PASS -- fast and brute-force bootstraps agree to 6 decimal places.


### Verdict rule (fixed before running the full sweep)

- **`supported`**: the 99% bootstrap CI excludes zero, AND the corresponding notebook `05d`
  seed ranges for the two models do not overlap.
- **`supported on this test set, not robust to training seed`**: the CI excludes zero, but the
  05d seed ranges DO overlap -- the gap is real on this specific test set and these specific
  fitted models, but a different training seed could plausibly have erased it.
- **`not supported`**: otherwise (the CI includes zero, regardless of the seed ranges).

05d's seed ranges are read directly from `results/results_ledger.csv` (raw-space R2,
`is_seed_sweep=True`, `variant="seed_sweep"` -- i.e. each combination's already-adopted XGBoost
`tree_method`, not the alternate-tree-method or reduced-sample diagnostic variants).

### Cost classification and the pooled-5-seed mechanism

F-DATA RandomForest (all three targets) and F-DATA power's XGBoost (`tree_method="exact"`) are
"expensive" -- refitting 5 times costs 20+ minutes per fit, not worth it for this test. These use
the canonical seed=0 model only (loaded from its already-saved fitted model object where one
exists, else refit once at seed=0), and any comparison touching one of these models is
restricted to seed=0 for BOTH models in the pair and flagged `expensive_seed0_only=True` --
pooling a 5-seed model against a 1-seed model would conflate seed-variance into what is supposed
to be a test-sampling estimate.

Every other model in every other combination is "cheap": refit at seeds 0-4, with each seed's own
2000-iteration cluster bootstrap pooled into one 10,000-draw distribution for the reported CI.
This reflects seed-to-seed spread on top of test-set sampling noise without anointing one
arbitrary seed's fit as canonical. Per-seed mean deltas are reported alongside the pooled result
for every such comparison, and any comparison where the per-seed deltas don't agree on sign is
flagged explicitly -- that is a materially weaker result than one where every seed points the
same way, even if the pooled interval happens to exclude zero.

### Top-5-user concentration, all six combinations

Before looking at any pairwise comparison: how much of each test set is a handful of heavy
users, by both row count and by sum-of-squares (the quantity that actually drives R2, via
SS_res/SS_tot). A user-level bootstrap result is suspect if a few users dominate it -- this is
checked once per combination up front, not discovered piecemeal per comparison.

In [7]:
fdata_user_ids = joblib.load(f"{INTERIM_DIR}/notebook08_fdata_user_ids.joblib")
pm100_user_ids = joblib.load(f"{INTERIM_DIR}/notebook08_pm100_user_ids.joblib")

COMBO_META = {
    "pm100_power": ("pm100_power_prepared.joblib", pm100_user_ids["pm100_power"]),
    "pm100_duration": ("pm100_duration_prepared.joblib", pm100_user_ids["pm100_duration"]),
    "pm100_memory_c4": ("pm100_memory_clean4_prepared.joblib", pm100_user_ids["pm100_memory_c4"]),
    "fdata_power": ("fdata_power_prepared.joblib", fdata_user_ids["fdata_power"]),
    "fdata_duration": ("fdata_duration_r2_prepared.joblib", fdata_user_ids["fdata_duration"]),
    "fdata_memory": ("fdata_memory_prepared.joblib", fdata_user_ids["fdata_memory"]),
}

share_rows = []
for combo, (fname, uids) in COMBO_META.items():
    p = joblib.load(f"{INTERIM_DIR}/{fname}")
    y_raw = p["y_test_raw"]
    stats = compute_per_user_stats(uids, y_raw, y_raw, y_raw)
    row_share, ss_share = top5_user_share(stats)
    share_rows.append(dict(combo=combo, n_unique_users=stats["n_users"], n_rows=stats["n_rows"],
                            top5_row_share=row_share, top5_ss_share=ss_share))

share_df = pd.DataFrame(share_rows).set_index("combo")
display(share_df.style.format({"n_rows": "{:,}", "top5_row_share": "{:.1%}",
                                "top5_ss_share": "{:.1%}"}))

,n_unique_users,n_rows,top5_row_share,top5_ss_share
combo,,,,
pm100_power,250,"54,093",79.7%,23.8%
pm100_duration,250,"54,093",79.7%,10.4%
pm100_memory_c4,250,"54,093",79.7%,53.9%
fdata_power,1419,"6,992,514",21.1%,0.1%
fdata_duration,1419,"6,992,514",21.1%,6.7%
fdata_memory,1419,"6,992,514",21.1%,21.9%


PM100's three combinations share an identical test-set split (no target-specific row-dropping
step in any of the three pipelines), so all three show the same 79.7% top-5-user row-share --
but their sum-of-squares shares diverge sharply by target (23.8% power, 10.4% duration, 53.9%
memory), since sum-of-squares depends on each target's own y-values, not on the row split. F-DATA
shows the opposite pattern: much lower row concentration (21.1%, spread across 1,419 users vs.
PM100's 250) but its own target-dependent sum-of-squares spread (0.1% power, 6.7% duration, 21.9%
memory). Both memory targets are the most concentrated by sum-of-squares in their respective
datasets -- worth keeping in mind when reading memory's bootstrap results below, since a
"user-level" result there leans more heavily on a handful of heavy users' residuals than any
other combination here.

### Full 18-comparison sweep

Computed by `scripts/notebook08_paired_bootstrap/run_full_sweep.py` (not re-run in this notebook
-- the RandomForest refits alone take ~20 minutes each, twice, for F-DATA duration and memory).
Every seed-0 prediction's R2 was checked against the real reported value during that run (same
Check-1-style guard as above, at 1e-6 tolerance for the same multi-threaded float-precision
reason), and every check passed before any bootstrap ran. Loading the saved results here.

In [8]:
results_df = pd.read_json(f"{RESULTS_DIR}/notebook08_bootstrap_results.json")
print(f"{len(results_df)} rows ({results_df['combo'].nunique()} combinations x 3 model pairs x "
      f"2 spaces)")

display_cols = ["combo", "model_a", "model_b", "space", "expensive_seed0_only", "n_seeds_used",
                 "mean_delta", "ci_lo", "ci_hi", "sign_disagreement", "verdict"]
display(results_df[display_cols].style.format({
    "mean_delta": "{:.4f}", "ci_lo": "{:.4f}", "ci_hi": "{:.4f}",
}))

36 rows (6 combinations x 3 model pairs x 2 spaces)


,combo,model_a,model_b,space,expensive_seed0_only,n_seeds_used,mean_delta,ci_lo,ci_hi,sign_disagreement,verdict
0,pm100_power,RandomForest,XGBoost,raw,False,5,-0.0219,-0.0484,0.0022,False,not supported
1,pm100_power,RandomForest,XGBoost,log,False,5,-0.0066,-0.0351,0.0001,False,not supported
2,pm100_power,RandomForest,LightGBM,raw,False,5,-0.0235,-0.0612,0.0031,False,not supported
3,pm100_power,RandomForest,LightGBM,log,False,5,-0.0006,-0.0113,0.0102,True,not supported
4,pm100_power,XGBoost,LightGBM,raw,False,5,-0.0015,-0.0143,0.0222,True,not supported
5,pm100_power,XGBoost,LightGBM,log,False,5,0.0062,-0.0021,0.0420,False,not supported
6,pm100_duration,RandomForest,XGBoost,raw,False,5,0.0009,-0.0678,0.0974,True,not supported
7,pm100_duration,RandomForest,XGBoost,log,False,5,0.0856,-0.0593,0.5185,False,not supported
8,pm100_duration,RandomForest,LightGBM,raw,False,5,-0.0911,-0.1520,-0.0294,False,supported
9,pm100_duration,RandomForest,LightGBM,log,False,5,-0.0662,-0.3047,0.0192,False,not supported


#### Comparisons where per-seed deltas don't agree on sign

Flagged per point 1 of the spec: even if the pooled interval excludes zero, a comparison where
individual seeds point in different directions is a materially weaker result than one where
every seed agrees, because it means the sign of the "winner" itself is not stable seed-to-seed
(a distinct, stronger instability than just the interval being wide).

In [9]:
sign_disagree_df = results_df[results_df["sign_disagreement"]][
    ["combo", "model_a", "model_b", "space", "mean_delta", "per_seed_mean_delta", "verdict"]
]
if len(sign_disagree_df) == 0:
    print("None -- every pooled comparison's individual seeds agreed on sign.")
else:
    print(f"{len(sign_disagree_df)} comparison(s) with per-seed sign disagreement:")
    display(sign_disagree_df)

3 comparison(s) with per-seed sign disagreement:


,combo,model_a,model_b,space,mean_delta,per_seed_mean_delta,verdict
3,pm100_power,RandomForest,LightGBM,log,-0.000556,"[-0.00032882800000000003, -0.0025869811, -0.0030291211, ...",not supported
4,pm100_power,XGBoost,LightGBM,raw,-0.001548,"[-0.0002756686, -0.0047796633000000005, -0.0064232398, 0...",not supported
6,pm100_duration,RandomForest,XGBoost,raw,0.000885,"[-0.0139458514, 0.0191355996, 0.0031517315000000002, 0.0...",not supported


#### Comparisons with a verdict of `supported` or `supported on this test set, not robust to training seed`

Every other comparison (29 of 36 rows) came back `not supported` -- the 99% interval included
zero. These are the exceptions.

In [10]:
notable = results_df[results_df["verdict"] != "not supported"][
    ["combo", "model_a", "model_b", "space", "mean_delta", "ci_lo", "ci_hi",
     "expensive_seed0_only", "verdict"]
].sort_values(["combo", "space"])
display(notable.style.format({"mean_delta": "{:.4f}", "ci_lo": "{:.4f}", "ci_hi": "{:.4f}"}))

,combo,model_a,model_b,space,mean_delta,ci_lo,ci_hi,expensive_seed0_only,verdict
31,fdata_memory,RandomForest,XGBoost,log,0.0140,0.0034,0.0286,True,"supported on this test set, not robust to training seed"
32,fdata_memory,RandomForest,LightGBM,raw,-0.0416,-0.0824,-0.0073,True,supported
22,fdata_power,XGBoost,LightGBM,raw,0.0033,0.0001,0.0062,True,supported
8,pm100_duration,RandomForest,LightGBM,raw,-0.0911,-0.1520,-0.0294,False,supported
10,pm100_duration,XGBoost,LightGBM,raw,-0.0926,-0.2206,-0.0288,False,supported
13,pm100_memory_c4,RandomForest,XGBoost,log,0.0050,0.0006,0.0180,False,supported
15,pm100_memory_c4,RandomForest,LightGBM,log,0.0027,0.0001,0.0102,False,"supported on this test set, not robust to training seed"


### Limitation

The test set in every combination is one contiguous chronological block (the last 30% of the
data by submission time, per `chronological_split`). User-level resampling here reflects
uncertainty about WHICH users appear and how much of the test period they're active for, but it
cannot capture correlation across users over time -- e.g. a cluster-wide event that shifted many
users' job characteristics simultaneously within that block would affect every bootstrap
resample identically, since resampling users doesn't resample the time period itself. This is a
property of the chronological-split design (deliberately used throughout this project to avoid
leaking future information into training), not a flaw specific to this bootstrap -- but it means
the reported intervals understate uncertainty relative to a hypothetical repeated-sampling-over-time
design, and should be read as "uncertainty given this test period," not "uncertainty over all
possible test periods."

### Summary

In [11]:
n_total = len(results_df)
n_supported = (results_df["verdict"] == "supported").sum()
n_not_robust = (results_df["verdict"] == "supported on this test set, not robust to training seed").sum()
n_not_supported = (results_df["verdict"] == "not supported").sum()
n_sign_disagree = results_df["sign_disagreement"].sum()
n_expensive = results_df["expensive_seed0_only"].sum()

print(f"{n_total} comparisons (18 model pairs x raw/log space):")
print(f"  supported: {n_supported}")
print(f"  supported on this test set, not robust to training seed: {n_not_robust}")
print(f"  not supported: {n_not_supported}")
print(f"  seed0-only (expensive model involved): {n_expensive} / {n_total}")
print(f"  pooled comparisons with per-seed sign disagreement: {n_sign_disagree}")
print()
print("Per-combination verdict counts:")
display(results_df.groupby(["combo", "verdict"]).size().unstack(fill_value=0))

36 comparisons (18 model pairs x raw/log space):
  supported: 5
  supported on this test set, not robust to training seed: 2
  not supported: 29
  seed0-only (expensive model involved): 14 / 36
  pooled comparisons with per-seed sign disagreement: 3

Per-combination verdict counts:


verdict,not supported,supported,"supported on this test set, not robust to training seed"
combo,,,
fdata_duration,6,0,0
fdata_memory,4,1,1
fdata_power,5,1,0
pm100_duration,4,2,0
pm100_memory_c4,4,1,1
pm100_power,6,0,0


Most pairwise model differences across all six target/dataset combinations are NOT
distinguishable from finite-test-set sampling noise at the 99% level, despite F-DATA's test sets
running into the millions of rows -- R2 gaps between two well-tuned tree ensembles on the same
features are simply often small relative to how much a single fixed test set's own composition
can move R2 under resampling. Where a gap IS supported, it is worth checking against 05d's own
seed-variance finding for the same pair before treating it as a robust, seed-independent claim --
that cross-check is exactly what separates `supported` from `supported on this test set, not
robust to training seed` in the verdict rule above. Raw-space and log-space did disagree on
verdict for several comparisons in both directions (not only PM100 duration, which is the combo
that originally motivated reporting both spaces) -- reporting both, rather than picking one, was
the right call.

Full results and the sweep script live in `scripts/notebook08_paired_bootstrap/` (`bootstrap_lib.py`
for the core machinery, `reconstruct_fdata_users.py` / `reconstruct_pm100_users.py` for the
user-identity reconstruction, `run_full_sweep.py` for the driver); the raw output table is
`results/notebook08_bootstrap_results.json`.